# Budgeted specification search

This page is for readers who have fitted a
model with `gamboost` ([Quickstart](quickstart.ipynb)) and read
[Choosing terms from the data](choosing_terms.ipynb), and who want to compare many
specifications under a fit budget. It documents `auto_policy`, `modeler_loop` and
`open_search`: what each does, what its scores and budgets measure, and how to read
the decision record it returns. The [Methodology](../methodology/overview.ipynb) part derives the methods used here.

`auto_policy` chooses among increasingly flexible predictive specifications under
a requested score and resource budget. `modeler_loop` adds repeated diagnostics
and an optional judgment callback that can propose domain edits. Both return a
fitted model and a decision record. Use the automatic policy for a fixed candidate
sequence; use the modeler loop when domain reasoning should propose changes
between measured comparisons.

The worked analyses [](../cases/budgeted_interaction_search.ipynb) and
[](../cases/bodyfat_modeler.ipynb) include independent evaluation data, annotated
decision tables and prediction curves.

## Automatic policy: `auto_policy`

`family` sets the loss used to fit models. `metric` sets the score used to choose
stopping iterations and adopt complete policy candidates. Supported response
scores are `mse`, `brier`, `log_loss`, `pinball` and `expectile`; the last two need
`score_parameter` between zero and one. `metric="family"` uses family risk.
The discovery stage, which screens main-effect terms before tuning and comparison, still ranks its candidates by family loss, so changing `metric` does
not yet make all intermediate probes optimize that score.

The following sample has a continuous response and uses Gaussian fitting with
MSE comparison. Its 80-fit budget covers discovery, tuning and comparisons.

In [1]:
import numpy as np
import polars as pl
import mboostrs as mb

rng = np.random.default_rng(5)
x = rng.uniform(-1, 1, 100)
data = pl.DataFrame({"x": x, "y": x*x + rng.normal(0, .1, 100)})
result = mb.auto_policy(
    data, "y", family=mb.Gaussian(), metric="mse", folds=3,
    discovery_fits=30, preexploration="dual", budget=mb.CostBudget(fits=80),
)
print(result.spec)
print(f"Fits: {result.fits}; score: {result.score}; stop: {result.stopped_early}")

bbs(x, df=4.0)
Fits: 48; score: 0.010580042802980328; stop: None


/Users/fcela/src/public_github/mboostrs/python/mboostrs/gamboost.py:3185: UserWarning: optimal mstop 99 is within 5% of the search cap 99; the search range was exhausted — consider increasing mstop and re-running cvrisk.
  result = CvRiskResult(


`result.model.predict(newdata, type="response")` predicts from the retained fit.
`result.trail` contains the candidate and baseline errors, paired gains, standard
errors and adoption decisions. `result.to_dict()` exports the record without model
internals. A fallback whose comparison did not finish has `score=None`; an
available prediction alone does not imply completed validation.

`complexity="simple"` limits the search to linear and adaptive main effects.
`"balanced"` also permits pairs and a residual tree with a two-SE gain rule;
`"rich"` uses a one-SE rule. These are search heuristics, not inferential tests.
Rejection retains the incumbent and lets later permitted stages try other
structure. The result's profile is `linear`, `additive`, `tensor`, `varying`
(for `by=` interactions without a tensor), or `hybrid` (an accepted residual tree).

## Preliminary exploration

`preexploration="native"` uses a boosted-tree probe; `"neural"` uses a small
neural network; `"dual"` uses both; and `"off"` skips these probes. The probes
nominate variables and interactions that subsequent fits can test. Neural
nominations do not establish that a nonlinear term belongs in the final model.
The combined mode charges two probe fits and can cost more without improving
the selected specification. See [](../methodology/preexploration.ipynb).

## Fold design and resource limits

Pass a `FoldScheme` through `folds=` to preserve its training weights and
held-out masks. With integer folds or no explicit design, `groups=` holds entire
declared groups out; multiple grouping columns keep connected rows together.
Random folds measure prediction for exchangeable new rows. Grouped and rolling
folds target different prediction tasks; the split should follow the deployment
setting. The policy's inner scores are descriptive after selection. Reserve
external test data before calling the search.

`CostBudget(fits=..., seconds=..., memory_mb=...)` covers the complete workflow.
Fit counts are hard admission limits. Time and memory are checked between native
calls and can overshoot while a call is running. Memory is the total peak resident set size (RSS, the physical memory the process holds) since the process started, in MiB. Failed native batches retain their reserved fit charges because
the exact completed-fold count is unavailable. `budget_report` records the
operations and blocked request. A zero-fit budget cannot produce an initial model
and raises a structured `workflow_budget` error.

`discovery_fits` is the automatic policy's main-effect probe allowance.
Pair screening reuses the accepted fit and a tree is proposed from its columns;
all candidate evaluations still count against the whole-workflow budget. `FitCache` can reuse an identical evaluated
specification across calls, keyed by data, folds, family, fitting arguments and
native build identity. This cache does not yet reuse prepared designs across
different specifications.

## Agent-directed session: `open_search`

`open_search(training, response, ...)` fixes the training frame, response,
fold scheme, score and budget once. It returns a `SearchSession` rather than a
chosen formula. `state()` lists legal next actions and their fit costs. An
analyst or external coding agent uses `run(action, ...)` to choose one test;
each `SearchEvidence` records its status, identifier, actual fit charges and
native result. The session does not require a provider key. Available actions
include classification, a marginal column screen, an optional native/neural
pre-scan, bounded `autospec` discovery, comparison, term report, fit/tuning,
residual pair screening and partial effects. A recorded hypothesis names what
would count against a proposed explanation before its outcome is read.

The example below uses the same training frame as the automatic-policy example.
It requests a smooth after writing a linear specification. The call measures
both candidates on shared folds; it does not silently adopt the lower number.

In [2]:
session = mb.open_search(
    data, "y", family=mb.Gaussian(), score="mse", folds=3,
    mstop=20, budget=mb.CostBudget(fits=20),
)
session.run("hypothesis", variable="x", role="measurement",
            statement="The x association may be curved.",
            would_count_against="A smooth increases held-out MSE.")
session.run("set_spec", rhs="bols(x)")
comparison = session.run("compare", candidates={"smooth": "bbs(x)"})
print([(row["specification"], round(row["out-of-fold risk"], 3))
       for row in comparison.value])
print(f"Fit charges: {session.state()['budget']['fit_charges']}")

[('current', 0.108), ('smooth', 0.014)]
Fit charges: 6


The caller may inspect `comparison.value`, request a term report, test a
competing hypothesis or adopt a candidate using the measured
`comparison.evidence_id`. A larger or reshaped candidate needs an
improvement beyond two paired standard errors. A smaller *nested*
specification may instead be adopted with `simplification_se=2` when its
error increase is within two paired standard errors. The default `0`
allows only a non-worsening removal. A proposal that fails the rule
remains exploratory. The
[commuting-zone case](../cases/cz_agent_search.ipynb) shows why choosing the
next diagnostic matters when smoking, Medicare spending and area composition
have different substantive roles.

`compare` reports every candidate that clears the two-SE rule in
`summary["two_se_candidates"]`, ordered by lower score change, then term count,
then name. `also_cleared` retains the runners-up; the order does not make the
choice for the caller. `soft_removal_candidates` lists smaller nested formulas
within the two-SE tolerance. An unrelated smaller formula cannot use this
removal rule.

`checkpoint()` is JSON-safe and `SearchSession.resume(training, checkpoint,
family=...)` checks the training data, native build, family parameters, fold
identity, evidence chain and costs. The fitted native model is not serialized;
refit explicitly after resume before inspecting residuals or freezing.
`export_transcript()` is a separate JSON-only record that remains available
after `freeze` and `final_evaluate`. It contains evidence, costs and the final
score without training rows, fold arrays or native model bytes; it cannot be
resumed for fitting.
`freeze` locks the chosen specification and `final_evaluate(data=test)` scores
one untouched sample once. A caller is responsible for withholding test rows
from earlier actions. A final family-risk score is not yet supported by this
one-time evaluator; choose a supported response score such as MSE or log loss.

With `causal=CausalSpec(...)`, the session locks declared baseline adjustment
covariates, excludes treatment and post-treatment columns from predictor
screens, and offers `causal_traps` as an explicit design check. This check
reports conflicts and missing support; it does not infer a causal effect.
The commuting-zone case is predictive and does not declare a treatment.

The session exposes an agent-usable path. `session.drive(judgment,
max_actions=...)` also lets a bounded callable choose successive public
actions. `LLMJudge` accepts a search-action digest when used as that callable;
its provider audit hashes and usage are attached to the session action record.
The driver refuses stale state IDs and excludes `final_evaluate`, so the
analyst still freezes and evaluates the test sample explicitly. Provider
credentials and calls are optional. `auto_policy` and `modeler_loop` retain
their fixed choices of which diagnostic comes next. Their measured fits,
screens, comparisons and tuning route through the shared session actions,
and their results expose `action_evidence`. The policy's budgeted fallback
fit and pair/tree nominations appear in that record. The modeler still
runs its initial autospec discovery before opening its session, and its
edit-choice rule is fixed. These routes therefore do not yet have identical
complete action trails. Do not read a fixed-loop transcript as evidence that an LLM chose
which diagnostic to run next.

## Screening and response-scale probes

`transform_candidates` runs a native pre-scan (one fitted surrogate in native
mode, two in dual mode) and returns advisory leads. A nonnegative, skewed
predictor may get a `bols(log1p(x))` sibling; a step-like pair may get
`btree(z, by=x, maxdepth=1)`, which learns one split rather than requiring a
hand-picked threshold. With a numeric `x` the term is a slope on `x` that
takes one value on each side of a cut point in `z`: the tree is grown on the residual divided by `x`, with each row weighted by the square of `x`, so each side's slope is the least-squares fit of the residual on `x`. With a 0/1 `x` it is mboost's
`btree(z, by = x)`, a tree grown on the rows where `x` is 1. Each lead includes an editable `candidate_rhs`; run
`compare` on shared folds before adopting it. Passing a prior `pre_scan` report
reuses that evidence with no fresh surrogate fit. A residual
`interaction_screen` marks thresholded siblings, but its `removed` score
is a half-split residual MSE, not a paired two-SE comparison.

`term_report(proxy_hygiene=True)` annotates a term with small conditional gain
when its raw numeric column is highly correlated with a locked sibling. It
returns a nested `drop_candidate` for a later `compare`; the annotation neither
deletes the term nor establishes confounding. `term_report(concurvity=True)`
refits the current specification by REML and reports every term whose
concurvity (the worst-case measure of the R package `mgcv`, which fits generalized additive models, from 0 to 1, where 1 means the other terms can reproduce the term exactly) is at least 0.8: such a
term's partial effect and the others' are not separately identified. It too
returns a finding and leaves the specification unchanged. The same report's
`concurvity_pairs` names which two terms are duplicates of each other, from
the boosted fit's pairwise measures (`concurvity(pairwise=True)`): one record
per unordered pair, at most five, each naming the pair, mgcv's three measures
and an interpretation. A term-vs-rest row says the term overlaps the rest as
a whole; a pair record says which other term carries it.
`pair_candidates(surrogate=True)`
runs a fresh native surrogate, while `surrogate=pre_scan_evidence.value` reuses
a retained report. Both return nominations, not adopted interactions.

For a positive, skewed Gaussian response scored by MSE, `response_probe`
compares identity with `log` or `log1p` on the *original response scale* and
charges one fit per candidate and fold. `log` requires positive observations;
`log1p` requires nonnegative observations, and this probe uses a fixed
`mstop`. Its back-transformed log prediction is a point prediction, not a
bias-corrected conditional mean. A spread hint from `transform_candidates`
points to explicit location-scale modeling when residual variance changes;
no action silently changes the family or the accepted specification. These
comparisons are exploratory after adaptive nomination.


In [3]:
positive = pl.DataFrame({
    "x": np.linspace(0, 2, 80),
    "y": np.expm1(1 + .6 * np.linspace(0, 2, 80)),
})
probe = mb.open_search(positive, "y", family=mb.Gaussian(), score="mse",
                       folds=3, mstop=30, budget=mb.CostBudget(fits=6))
probe.run("set_spec", rhs="bols(x)")
response_options = probe.run("response_probe", transforms=("identity", "log1p"))
print([(row["transform"], round(row["mse"], 3))
       for row in response_options.value])
print("Native fits charged:", response_options.fit_charges)


[('identity', 0.09), ('log1p', 0.008)]
Native fits charged: 6


## Judgment loop: `modeler_loop`

`modeler_loop(data, response, family=..., judgment=...)` runs discovery, term
reports, interaction screening, proposed edits, measured comparisons and final
tuning. `family=` fixes the family; omitting it permits inference by `autospec`.
Its comparisons currently use family risk. The loop does not expose the automatic
policy's requested-score or explicit-fold interface. Its automatic group detection
and the handling of missing values should be checked before use on dependent or missing data (rows with a missing or non-finite value in a modelled numeric column are dropped before the search); the automatic policy's broader controls do not carry over implicitly.

`budget` limits initial probes, `max_rounds` limits edit rounds, and
`workflow_budget=CostBudget(...)` limits the entire loop. `judgment=None` runs the
deterministic reference search. `ModelerResult.model` is the retained fit,
`stop_reason` explains termination, and `transcript.steps` records evidence,
judgments, comparisons and accepted edits. `to_dict()` exports this record.

## Editable judgments: `SpecificationJudgment`

A callback receives an aggregate digest with the current specification, term
gains, interaction leads, locks, remaining resources and an `evidence_id`.
Return a `SpecificationJudgment(...).to_dict()` or the equivalent mapping.
Allowed edits are `drop`, `add`, `replace`, `lock` and monotonicity constraints.
A replacement maps an exact current term to its replacement. A monotonicity edit
names a column and `"increasing"` or `"decreasing"`. Copy the digest's evidence
identifier so the proposed edit is attached to the measured state.

In [4]:
proposal = mb.SpecificationJudgment(
    drop=("unused_measurement",),
    lock=("baseline_measurement",),
    reasoning="Test whether the optional measurement is dispensable.",
)
print(proposal.to_dict())

{'drop': ['unused_measurement'], 'add': [], 'lock': ['baseline_measurement'], 'replace': {}, 'constraints': {'monotone': {}}, 'flags': [], 'assumptions': [], 'reasoning': 'Test whether the optional measurement is dispensable.', 'estimand_note': ''}


The names in this illustrative proposal must be replaced with predictors in the
current data. Unknown or forbidden columns, stale evidence identifiers, malformed
formulas and edits that remove locked columns are refused. A valid proposal is a
candidate for measurement, not an instruction to adopt it. More complex edits
need a two-SE improvement; removals may retain a smaller model when the fold
comparison cannot distinguish it. The body-fat case displays both the proposed
edit and the numerical decision.

## Provider-backed judgments: `LLMJudge`

`LLMJudge(llm=..., base_url=..., api_key=...)` connects the callback to a configured
chat provider. A compatible `client=` can supply its own connection. This optional
path sends the training-evidence digest and domain context; it is not needed for
deterministic searches. The [body-fat extension](../cases/bodyfat_modeler.ipynb)
shows a credential-configured call that is not executed by the book.

`max_calls` bounds provider requests. `max_tokens` bounds the requested response
length; `max_total_tokens` is checked between requests from reported usage and
can be exceeded by the final request. Missing usage blocks subsequent requests
under that limit. `calls` records model/provider identity, hashes, usage and
failures; `retain_text=True` additionally retains prompt and reply text. Provider
failure or invalid output is logged and leaves deterministic candidates available.

## Interpretation and scope

A search score cannot establish causality, calibrated uncertainty after selection,
or superiority over unseen data sets. The current examples demonstrate prediction
and auditability. Causal workflows require a separately stated estimand and design;
see [](causal.ipynb). Provider-backed task-study performance and complete
whole-workflow performance benchmarks remain to be established.

The paired-search calculation and its evaluation boundary are derived in
[](../methodology/search_decisions.ipynb).